# Dataset Conversion

In [ ]:
# ============================================================
# ✏️  Configuration — edit these values
# ============================================================

DATA_ROOT  = "/home/jvermandere/datasets/V-Scan/data"
OUT_ROOT   = "/home/jvermandere/datasets/V-Scan/mmdet3d"
CLASS_NAME = "object"
VAL_RATIO  = 0.2   # fraction of scenes held out for validation
SEED       = 42

# ============================================================
# 🚀  Run conversion  (no edits needed below)
# ============================================================

import sys
sys.path.insert(0, '../')
import drm
import json
import math
import pickle
import numpy as np
from pathlib import Path


# ---------------------------------------------------------------------------
# OBB extraction from 8 corners
# ---------------------------------------------------------------------------

def corners_to_box7(corners_8x3: np.ndarray) -> np.ndarray:
    corners = corners_8x3.astype(np.float64)
    center  = corners.mean(axis=0)
    dy      = corners[:, 1].max() - corners[:, 1].min()

    xz          = corners[:, [0, 2]]
    xz_centered = xz - xz.mean(axis=0)
    cov         = (xz_centered.T @ xz_centered) / len(xz_centered)
    evals, evecs = np.linalg.eigh(cov)
    main_axis   = evecs[:, 1]
    yaw         = math.atan2(float(main_axis[0]), float(main_axis[1]))

    cos_y, sin_y = math.cos(yaw), math.sin(yaw)
    local_x = xz_centered @ np.array([cos_y,  sin_y])
    local_z = xz_centered @ np.array([-sin_y, cos_y])
    dx = float(local_x.max() - local_x.min())
    dz = float(local_z.max() - local_z.min())

    return np.array([center[0], center[1], center[2], dx, dy, dz, yaw],
                    dtype=np.float32)


# ---------------------------------------------------------------------------
# Per-scene processing
# ---------------------------------------------------------------------------

def process_scene(scene_dir: Path, out_points_dir: Path,
                  class_name: str = "object") -> dict | None:
    scene_name = scene_dir.name
    bin_path   = scene_dir / "main.bin"
    json_path  = scene_dir / "main_bb.json"

    if not bin_path.exists():
        print(f"  [SKIP] {scene_name}: main.bin not found")
        return None
    if not json_path.exists():
        print(f"  [SKIP] {scene_name}: main_bb.json not found")
        return None

    # -- Point cloud ----------------------------------------------------------
    pts_raw = np.fromfile(str(bin_path), dtype=np.float32)
    cols = 6
    if pts_raw.size % cols != 0:
        if pts_raw.size % 3 == 0:
            cols = 3
            print(f"  [WARN] {scene_name}: no colour channels, using xyz only")
        else:
            print(f"  [SKIP] {scene_name}: .bin size not divisible by 6 or 3")
            return None

    pts = pts_raw.reshape(-1, cols)
    pts.tofile(str(out_points_dir / f"{scene_name}.bin"))
    print(f"  {scene_name}: {len(pts):,} points, {cols} dims")

    # -- Bounding boxes -------------------------------------------------------
    with open(json_path) as f:
        bb_data = json.load(f)

    instances = []
    for i, bb in enumerate(bb_data.get("boxes", [])):
        raw_corners = bb.get("boundingPoints", [])
        if len(raw_corners) != 8:
            print(f"  [WARN] box {i} has {len(raw_corners)} corners — skipped")
            continue

        corners_unity = np.array([[p["x"], p["y"], p["z"]] for p in raw_corners],
                                  dtype=np.float64)
        corners_scan  = drm.transform_points(corners_unity, drm.UNITY2TRIMESH_T)
        box7          = corners_to_box7(corners_scan)

        instances.append({
            "bbox_3d":       box7.tolist(),  # [cx, cy, cz, dx, dy, dz, yaw]
            "bbox_label_3d": 0,              # single class → always 0
        })

    print(f"           {len(instances)} instances")

    # -- Info dict (MMDetection3D v1.x format) --------------------------------
    # lidar_points['lidar_path'] is read by LoadPointsFromFile
    # instances is read by Det3DDataset.parse_ann_info()
    return {
        "lidar_points": {
            "num_pts_feats": cols,
            "lidar_path":    f"points/{scene_name}.bin",
        },
        "sample_id": scene_name,
        "instances": instances,
    }


# ---------------------------------------------------------------------------
# Main
# ---------------------------------------------------------------------------

data_root = Path(DATA_ROOT)
out_root  = Path(OUT_ROOT)

out_points_dir = out_root / "points"
out_points_dir.mkdir(parents=True, exist_ok=True)

scene_dirs = sorted([
    p for p in data_root.iterdir()
    if p.is_dir() and (p / "main.bin").exists()
])
print(f"Found {len(scene_dirs)} scenes under {data_root}\n")

if not scene_dirs:
    raise FileNotFoundError("No scenes found — check DATA_ROOT path.")

all_infos = {}
for sd in scene_dirs:
    print(f"→ Processing: {sd.name}")
    info = process_scene(sd, out_points_dir, CLASS_NAME)
    if info is not None:
        all_infos[sd.name] = info

# Train / val split
rng         = np.random.default_rng(SEED)
scene_names = np.array(sorted(all_infos.keys()))
rng.shuffle(scene_names)

n_val       = max(1, int(len(scene_names) * VAL_RATIO))
val_names   = scene_names[:n_val].tolist()
train_names = scene_names[n_val:].tolist()

print(f"\nSplit: {len(train_names)} train  /  {len(val_names)} val")

(out_root / "train_list.txt").write_text("\n".join(train_names) + "\n")
(out_root / "val_list.txt"  ).write_text("\n".join(val_names)   + "\n")

# Write pkl files in MMEngine v1.x format: {'metainfo': ..., 'data_list': [...]}
for split, names in [("train", train_names), ("val", val_names)]:
    data_list = [all_infos[n] for n in names]
    pkl = {
        'metainfo': {
            'classes': (CLASS_NAME,),
            'dataset': 'VScan',
            'split':   split,
        },
        'data_list': data_list,
    }
    pkl_path = out_root / f"vscan_infos_{split}.pkl"
    with open(pkl_path, "wb") as f:
        pickle.dump(pkl, f)
    print(f"Saved {len(data_list):>4} infos → {pkl_path}")

print(f"""
✓ Conversion complete.
  Output root : {out_root}
  Points      : {out_points_dir}
  Train pkl   : {out_root / 'vscan_infos_train.pkl'}
  Val   pkl   : {out_root / 'vscan_infos_val.pkl'}
""")

## Visualise to confirm

In [ ]:
# ============================================================
# ✏️  Configuration
# ============================================================

MMDET3D_ROOT = "/home/jvermandere/datasets/V-Scan/mmdet3d"
SCENE_NAME   = "bedroom_Leica-P30_1775809921180"   # or None to pick randomly
BB_COLOR     = [255, 0, 0, 200]   # RGBA

# ============================================================
# 🔍  Visualiser  (no edits needed below)
# ============================================================

import math
import pickle
import numpy as np
import trimesh
import trimesh.path.entities
from pathlib import Path

# ── Helpers ───────────────────────────────────────────────────────────────

def box7_to_corners(box7):
    cx, cy, cz, dx, dy, dz, yaw = box7
    hx, hy, hz = dx / 2, dy / 2, dz / 2
    local = np.array([
        [ hx, -hy,  hz], [-hx, -hy,  hz],
        [-hx, -hy, -hz], [ hx, -hy, -hz],
        [ hx,  hy,  hz], [-hx,  hy,  hz],
        [-hx,  hy, -hz], [ hx,  hy, -hz],
    ])
    c, s = math.cos(yaw), math.sin(yaw)
    Ry = np.array([[c, 0, s], [0, 1, 0], [-s, 0, c]])
    return (Ry @ local.T).T + np.array([cx, cy, cz])


def corners_to_wireframe(corners_8x3, color):
    c = corners_8x3.astype(np.float64)
    hull  = trimesh.convex.convex_hull(c)
    edges = set()
    for f in hull.faces:
        for i in range(3):
            edges.add(tuple(sorted([f[i], f[(i+1) % 3]])))
    entities = [trimesh.path.entities.Line(list(e)) for e in edges]
    return trimesh.path.Path3D(
        entities=entities,
        vertices=hull.vertices,
        colors=[color] * len(entities),
    )


def load_points(pts_path):
    pts_raw = np.fromfile(str(pts_path), dtype=np.float32)
    cols = 6 if pts_raw.size % 6 == 0 else 3
    pts  = pts_raw.reshape(-1, cols)
    xyz  = pts[:, :3]
    if cols >= 6:
        rgb = pts[:, 3:6]
        rgb = (rgb * 255).astype(np.uint8) if rgb.max() <= 1.01 else rgb.astype(np.uint8)
        colors = np.hstack([rgb, np.full((len(rgb), 1), 255, np.uint8)])
    else:
        colors = np.full((len(xyz), 4), [200, 200, 200, 255], np.uint8)
    return xyz, colors


def load_split_index(mmdet3d_root):
    """Build a dict {scene_name: (split, info)} from both train and val pkls."""
    index = {}
    for split in ("train", "val"):
        pkl_path = mmdet3d_root / f"vscan_infos_{split}.pkl"
        if not pkl_path.exists():
            continue
        with open(pkl_path, "rb") as f:
            raw = pickle.load(f)

        data_list = raw["data_list"] if isinstance(raw, dict) else raw

        for info in data_list:
            name = info["sample_id"]          # v1.x key
            index[name] = (split, info)
    return index


# ── Load index ────────────────────────────────────────────────────────────

root  = Path(MMDET3D_ROOT)
index = load_split_index(root)
print(f"Dataset: {len(index)} scenes total")
print(f"  train: {sum(1 for s,_ in index.values() if s=='train')}")
print(f"  val  : {sum(1 for s,_ in index.values() if s=='val')}\n")

# ── Pick scene ────────────────────────────────────────────────────────────

if SCENE_NAME is None:
    SCENE_NAME = np.random.choice(list(index.keys()))
    print(f"Randomly selected: {SCENE_NAME}")

if SCENE_NAME not in index:
    raise KeyError(f"Scene '{SCENE_NAME}' not found. Available:\n" +
                   "\n".join(f"  • {k}" for k in sorted(index)))

split, info = index[SCENE_NAME]
print(f"Scene : {SCENE_NAME}  [{split}]")

# ── Load points ───────────────────────────────────────────────────────────

pts_rel     = info["lidar_points"]["lidar_path"]   # v1.x key
pts_path    = root / pts_rel
xyz, colors = load_points(pts_path)
print(f"Points: {len(xyz):,}")

# ── Load boxes from instances ─────────────────────────────────────────────

instances = info.get("instances", [])
boxes7    = np.array([inst["bbox_3d"] for inst in instances], dtype=np.float32) \
            if instances else np.zeros((0, 7), dtype=np.float32)
print(f"Boxes : {len(boxes7)}\n")

for i, (inst, box) in enumerate(zip(instances, boxes7)):
    cx, cy, cz, dx, dy, dz, yaw = box
    label = inst["bbox_label_3d"]
    print(f"  [{i}] label={label}  "
          f"center=({cx:+.2f}, {cy:+.2f}, {cz:+.2f})  "
          f"size=({dx:.2f}×{dy:.2f}×{dz:.2f})  "
          f"yaw={math.degrees(yaw):.1f}°")

# ── Build trimesh scene ───────────────────────────────────────────────────

scene = trimesh.Scene()
scene.add_geometry(trimesh.PointCloud(vertices=xyz, colors=colors))

for i, box7 in enumerate(boxes7):
    corners   = box7_to_corners(box7)
    wireframe = corners_to_wireframe(corners, BB_COLOR)
    scene.add_geometry(wireframe, node_name=f"box_{i}")

print("\nOpening viewer…  (close the window to continue)")
scene.show()

## Training

In [ ]:
# ===========================================================================
# V-Scan VoteNet — Training Setup & Launch
# ===========================================================================
# Run each cell in order.  Cells marked 🔧 require you to set a path.

# ---------------------------------------------------------------------------
# Cell 1 — Paths  🔧
# ---------------------------------------------------------------------------

MMDET3D_ROOT = "/home/jvermandere/projects/mmdetection3d"   # your cloned repo
VSCAN_ROOT   = "/home/jvermandere/datasets/V-Scan/mmdet3d"
CONFIG_PATH  = f"{MMDET3D_ROOT}/configs/votenet/votenet_vscan.py"


In [ ]:
# ---------------------------------------------------------------------------
# Cell 2 — Install files into mmdetection3d
# ---------------------------------------------------------------------------
import shutil, os

# 1. Dataset class
shutil.copy(f"{VSCAN_ROOT}/vscan_dataset.py",
            f"{MMDET3D_ROOT}/mmdet3d/datasets/vscan_dataset.py")
print("✓ copied vscan_dataset.py")

# 2. Freeze hook
shutil.copy(f"{VSCAN_ROOT}/freeze_hook.py",
            f"{MMDET3D_ROOT}/mmdet3d/engine/hooks/freeze_hook.py")
print("✓ copied freeze_hook.py")

# 3. Config
os.makedirs(f"{MMDET3D_ROOT}/configs/votenet", exist_ok=True)
shutil.copy(f"{VSCAN_ROOT}/votenet_vscan.py", CONFIG_PATH)
print("✓ copied votenet_vscan.py")


In [ ]:

# ---------------------------------------------------------------------------
# Cell 3 — Register VScanDataset in mmdet3d/datasets/__init__.py
# ---------------------------------------------------------------------------

datasets_init = f"{MMDET3D_ROOT}/mmdet3d/datasets/__init__.py"

with open(datasets_init) as f:
    src = f.read()

IMPORT_LINE = "from .vscan_dataset import VScanDataset"
EXPORT_ITEM = "'VScanDataset'"

if IMPORT_LINE not in src:
    # Add import after the last existing 'from .' import
    lines = src.splitlines()
    last_from = max(i for i,l in enumerate(lines) if l.startswith("from ."))
    lines.insert(last_from + 1, IMPORT_LINE)
    src = "\n".join(lines)
    print("✓ added import line")
else:
    print("  import line already present")

if EXPORT_ITEM not in src:
    # Append to __all__ list
    src = src.replace("__all__ = [", f"__all__ = [\n    {EXPORT_ITEM},")
    print("✓ added to __all__")
else:
    print("  __all__ entry already present")

with open(datasets_init, "w") as f:
    f.write(src)


In [ ]:

# ---------------------------------------------------------------------------
# Cell 5 — Enable the freeze hook in the config
# ---------------------------------------------------------------------------
# The config has a placeholder custom_hooks list.  This cell replaces it
# with the real FreezeBackboneHook entry.

with open(CONFIG_PATH) as f:
    cfg_src = f.read()

OLD = """custom_hooks = [
    dict(
        type='NumClassCheckHook',   # built-in sanity check
    ),
]"""

NEW = """custom_hooks = [
    dict(type='NumClassCheckHook'),
    dict(type='FreezeBackboneHook', freeze_epochs=10),
]"""

if "FreezeBackboneHook" not in cfg_src:
    cfg_src = cfg_src.replace(OLD, NEW)
    with open(CONFIG_PATH, "w") as f:
        f.write(cfg_src)
    print("✓ FreezeBackboneHook added to config")
else:
    print("  already present in config")


In [ ]:
# ---------------------------------------------------------------------------
# Cell 6 — Sanity-check: verify the dataset loads correctly
# ---------------------------------------------------------------------------
import sys, pickle
from pathlib import Path

sys.path.insert(0, MMDET3D_ROOT)

for split in ("train", "val"):
    pkl = Path(VSCAN_ROOT) / f"vscan_infos_{split}.pkl"
    with open(pkl, "rb") as f:
        raw = pickle.load(f)

    # MMEngine v1.x dict format
    infos = raw["data_list"] if isinstance(raw, dict) else raw

    boxes = sum(len(i["instances"]) for i in infos)
    print(f"  {split:>5}: {len(infos):>4} scenes  |  {boxes:>5} boxes total  "
          f"|  avg {boxes/max(len(infos),1):.1f} boxes/scene")

In [ ]:
# Cell 7 — Launch training 🚀

import subprocess, sys
import os
os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":4096:8"
# or alternatively:
# os.environ["CUBLAS_WORKSPACE_CONFIG"] = ":16:8"

cmd = [
    sys.executable,
    f"{MMDET3D_ROOT}/tools/train.py",
    CONFIG_PATH,
    "--work-dir", f"{MMDET3D_ROOT}/work_dirs/votenet_vscan",
]

print("Running:\n  " + " ".join(cmd) + "\n")
subprocess.run(cmd, check=True)

In [ ]:

# ---------------------------------------------------------------------------
# Cell 8 — Monitor training loss (TensorBoard)
# ---------------------------------------------------------------------------
# Uncomment the tensorboard hook in votenet_vscan.py first, then run:
#
%load_ext tensorboard
%tensorboard --logdir {MMDET3D_ROOT}/work_dirs/votenet_vscan

#tensorboard --logdir /home/jvermandere/projects/mmdetection3d/work_dirs/votenet_vscan --port 6006

In [ ]:
# ---------------------------------------------------------------------------
# Cell 9 — Evaluate best checkpoint
# ---------------------------------------------------------------------------
import glob, os

work_dir = f"{MMDET3D_ROOT}/work_dirs/votenet_vscan"

# Find the best checkpoint — filename is e.g. best_mAP_0.25_epoch_25.pth
best_ckpts = sorted(glob.glob(f"{work_dir}/best_indoor_mAP_0.25_epoch_*.pth"))
latest_ckpt = f"{work_dir}/latest.pth"

if best_ckpts:
    CHECKPOINT = best_ckpts[-1]   # highest epoch = best so far
    print(f"Using best checkpoint: {os.path.basename(CHECKPOINT)}")
else:
    CHECKPOINT = latest_ckpt
    print(f"No best checkpoint found, using: latest.pth")

cmd = [
    sys.executable,
    f"{MMDET3D_ROOT}/tools/test.py",
    CONFIG_PATH,
    CHECKPOINT,
]

print("Running:\n  " + " ".join(cmd) + "\n")
subprocess.run(cmd, check=True)

In [ ]:
# ---------------------------------------------------------------------------
# Cell 10 — Copy best checkpoint to weights folder
# ---------------------------------------------------------------------------
import glob, os, shutil

work_dir  = f"{MMDET3D_ROOT}/work_dirs/votenet_vscan"
out_dir   = "/home/jvermandere/projects/DRM/_weights"
out_name  = "votenet_vscan_best.pth"

best_ckpts = sorted(glob.glob(f"{work_dir}/best_indoor_mAP_0.25_epoch_*.pth"))

if not best_ckpts:
    print("No best checkpoint found — has validation run yet?")
else:
    src = best_ckpts[-1]
    dst = os.path.join(out_dir, out_name)
    shutil.copy(src, dst)
    print(f"Copied: {os.path.basename(src)}")
    print(f"    → {dst}")